# 04_dim_playlists

DML: dim_playlists — Playlist dimension, PK: playlist_id.

In [ ]:
%run ../../tools/config/settings
%run ../../tools/delta/upsert

In [ ]:
dbutils.widgets.text("run_id",         "")
dbutils.widgets.text("ingestion_date", "")
run_id         = dbutils.widgets.get("run_id")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [ ]:
from pyspark.sql import functions as F

src = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_playlists")
    .filter(F.col("ingestion_date") == ingestion_date)
    .filter(F.col("run_id") == run_id)
    .select(
        "playlist_id", "playlist_name", "description",
        "owner_id", "owner_name", "total_tracks",
        "is_public", "is_collaborative", "ingestion_date",
    )
    .withColumn("_valid_from", F.col("ingestion_date"))
    .drop("ingestion_date")
    .dropDuplicates(["playlist_id"])
)

upsert_delta(src, f"{CATALOG}.{SILVER_SCHEMA}.dim_playlists", ["playlist_id"])
print(f"dim_playlists: {src.count()} rows upserted")